# AI Virtual Assistant for Customer Service - LaunchPad

This notebook deploys the NVIDIA AI Virtual Assistant blueprint on an NVIDIA LaunchPad instance with 2 H100 GPUs.

Run this notebook from **Resources > Code Server IDE** after cloning the repository. The LaunchPad **Jupyter Notebook** resource is a separate managed container and is not the target environment for this notebook.


## What This Notebook Starts

- Nemotron 3 Nano local NIM on GPU 0
- Embedding NIM, reranking NIM, and GPU Milvus on GPU 1
- Prebuilt GHCR application containers for the agent, retrievers, analytics, API gateway, and UI
- A LaunchPad UI proxy on host port `3001` for the VS Code **Ports** tab

You need an NGC personal API key with access to NIM containers and model assets. You do not need a hosted NVIDIA API Catalog key for inference in this LaunchPad path.


## Before You Run The Cells

Confirm that you are in the Code Server IDE and that the repository has been cloned:

```bash
cd ~/ai-virtual-assistant
git branch --show-current
```

Expected branch:

```text
nemotron3-milvus-cpu
```

If the instance was prepared for notebooks, select the `AIVA LaunchPad` kernel. The default Python kernel can run the deployment cells, but `AIVA LaunchPad` is also configured for the ingestion notebook.


In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import time
from collections import deque


def find_repo_root():
    start = Path.cwd().resolve()
    candidates = [start, *start.parents, Path.home() / "ai-virtual-assistant", Path("/home/nvidia/ai-virtual-assistant")]
    for candidate in candidates:
        if (candidate / "deploy" / "compose" / "docker-compose.yaml").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not find deploy/compose/docker-compose.yaml. Open this notebook from the ai-virtual-assistant repo in Code Server.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

ENV_FILE = REPO_ROOT / ".env.launchpad"
COMPOSE_FILES = [
    REPO_ROOT / "deploy" / "compose" / "docker-compose.yaml",
    REPO_ROOT / "deploy" / "compose" / "docker-compose.ghcr.yaml",
    REPO_ROOT / "launchpad" / "docker-compose.launchpad.yaml",
]

COMPOSE_ARGS = []
for compose_file in COMPOSE_FILES:
    if not compose_file.exists():
        raise FileNotFoundError(compose_file)
    COMPOSE_ARGS.extend(["-f", str(compose_file)])

LOG_DIR = REPO_ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)
DEPLOY_LOG = LOG_DIR / "ai_virtual_assistant_launchpad.log"

print(f"Repository root: {REPO_ROOT}")
print(f"Environment file: {ENV_FILE}")
print("Compose files:")
for compose_file in COMPOSE_FILES:
    print(f"- {compose_file}")


## Configure The LaunchPad Environment

If lab staff already prepared `.env.launchpad`, this cell reuses that key. Otherwise, paste the lab NGC personal API key when prompted. The notebook writes `.env.launchpad` from `launchpad/.env.example` and keeps it local to this LaunchPad instance.


In [ ]:
def env_file_value(path, key):
    if not path.exists():
        return ""
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.startswith(f"{key}="):
            return line.split("=", 1)[1].strip().strip('"')
    return ""


NGC_API_KEY = os.environ.get("NGC_API_KEY", "") or env_file_value(ENV_FILE, "NGC_API_KEY")
if not NGC_API_KEY or NGC_API_KEY == "<paste-ngc-api-key>":
    NGC_API_KEY = getpass.getpass("Enter your NGC personal API key: ")

if not NGC_API_KEY:
    raise ValueError("NGC_API_KEY is required for local NIM containers.")

template = (REPO_ROOT / "launchpad" / ".env.example").read_text(encoding="utf-8")
env_text = template.replace("NGC_API_KEY=<paste-ngc-api-key>", f"NGC_API_KEY={NGC_API_KEY}")
ENV_FILE.write_text(env_text, encoding="utf-8")
ENV_FILE.chmod(0o600)
os.environ["NGC_API_KEY"] = NGC_API_KEY

nim_cache = Path("/home/nvidia/.cache/nim")
nim_cache.mkdir(parents=True, exist_ok=True)

print(f"Wrote {ENV_FILE}")
print(f"NIM model cache: {nim_cache}")


## Check Docker And GPU Access

These cells should run in the Code Server environment on the LaunchPad host. If Docker or `nvidia-smi` is not available here, switch to **Resources > Code Server IDE** and use this notebook from there.


In [ ]:
def docker_cmd(*args):
    probe = subprocess.run(["docker", "info"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if probe.returncode == 0:
        return ["docker", *map(str, args)]
    return ["sudo", "docker", *map(str, args)]


def run_capture(command, input_text=None):
    result = subprocess.run(command, input=input_text, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"Command failed: {' '.join(map(str, command))}")
    return result.stdout.strip()


def run_logged(command, log_file, error_message):
    recent_lines = deque(maxlen=10)
    print(f"Running: {' '.join(map(str, command))}", flush=True)
    print(f"Streaming output to {log_file}", flush=True)
    last_progress = time.monotonic()

    with log_file.open("a", encoding="utf-8") as log:
        log.write(f"\n\n$ {' '.join(map(str, command))}\n")
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            log.write(line)
            log.flush()
            stripped = line.strip()
            if stripped:
                recent_lines.append(stripped)
            now = time.monotonic()
            if now - last_progress >= 10:
                print(".", end="", flush=True)
                last_progress = now
        return_code = process.wait()

    print(" done", flush=True)
    if return_code != 0:
        print(error_message)
        print(f"Full output: {log_file}")
        print("Last output lines:")
        for line in recent_lines:
            print(line)
        raise RuntimeError(f"Command failed with exit code {return_code}.")


print(run_capture(docker_cmd("compose", "version")))
try:
    print(run_capture(["nvidia-smi", "-L"]))
except Exception as exc:
    raise RuntimeError("nvidia-smi is not available from this notebook. Use the Code Server IDE on the LaunchPad host.") from exc


## Authenticate To NGC


In [ ]:
login = subprocess.run(
    docker_cmd("login", "nvcr.io", "-u", "$oauthtoken", "--password-stdin"),
    input=NGC_API_KEY,
    text=True,
    capture_output=True,
)
if login.returncode != 0:
    print(login.stdout)
    print(login.stderr)
    raise RuntimeError("Docker login to nvcr.io failed. Confirm this is an NGC personal API key with NIM access.")

print("Docker is authenticated with nvcr.io.")


## Validate The Compose Configuration

This checks that Docker Compose is configured for local Nemotron 3 Nano, local embedding/reranking NIMs, GPU Milvus, GHCR app images, and the LaunchPad UI proxy.


In [ ]:
config = subprocess.run(
    docker_cmd("compose", "--env-file", ENV_FILE, *COMPOSE_ARGS, "--profile", "local-nim", "config"),
    text=True,
    capture_output=True,
)
if config.returncode != 0:
    print(config.stdout)
    print(config.stderr)
    raise RuntimeError("Docker Compose config validation failed.")

rendered = config.stdout
assert "nvcr.io/nim/nvidia/nemotron-3-nano" in rendered
assert "nvidia/nemotron-3-nano-30b-a3b" in rendered
assert "nvcr.io/nim/nvidia/llama-nemotron-embed-1b-v2" in rendered
assert "nvcr.io/nim/nvidia/llama-nemotron-rerank-1b-v2" in rendered
assert "milvusdb/milvus" in rendered and "gpu" in rendered
assert "agent-frontend-proxy" in rendered
assert "18086" in rendered
assert "--enable-auto-tool-choice" in rendered
assert "--tool-call-parser qwen3_coder" in rendered
assert "\nbuild:" not in rendered

print("Docker Compose configuration validated for LaunchPad local NIMs, GPU Milvus, GHCR app images, and the UI proxy.")


## Start The Stack

This cell pulls missing prebuilt application, LaunchPad UI, and NIM images, then starts the stack with `--no-build`.

The first run can take a long time because the NIM containers download model assets into `/home/nvidia/.cache/nim`. Full output is written to `logs/ai_virtual_assistant_launchpad.log`.

The LaunchPad UI image is published to GHCR as `ghcr.io/jspaulding-nv/aiva-customer-service-ui:nemotron3-launchpad-proxy`. Set `AIVA_BUILD_LAUNCHPAD_UI=1` before running this notebook only if you intentionally want to rebuild it locally.


In [ ]:
DEPLOY_LOG.write_text("", encoding="utf-8")

build_ui = os.environ.get("AIVA_BUILD_LAUNCHPAD_UI", "0").strip().lower() in {"1", "true", "yes"}
if build_ui:
    run_logged(["bash", "launchpad/build-launchpad-ui.sh"], DEPLOY_LOG, "LaunchPad UI image build failed.")
else:
    print("Using the published LaunchPad UI image. Set AIVA_BUILD_LAUNCHPAD_UI=1 to rebuild locally.")

run_logged(
    docker_cmd("compose", "--env-file", ENV_FILE, *COMPOSE_ARGS, "--profile", "local-nim", "pull", "--policy", "missing"),
    DEPLOY_LOG,
    "Docker Compose image pull failed.",
)

run_logged(
    docker_cmd("compose", "--env-file", ENV_FILE, *COMPOSE_ARGS, "--profile", "local-nim", "up", "-d", "--no-build"),
    DEPLOY_LOG,
    "Docker Compose deployment failed.",
)

print("LaunchPad stack start requested.")
print(f"Full output: {DEPLOY_LOG}")


## Check Container Status

Some services may still be starting immediately after `up -d`. Rerun this cell while the NIMs download model assets and become healthy.


In [ ]:
ps = subprocess.run(
    docker_cmd("compose", "--env-file", ENV_FILE, *COMPOSE_ARGS, "--profile", "local-nim", "ps"),
    text=True,
    capture_output=True,
)
print(ps.stdout)
if ps.returncode != 0:
    print(ps.stderr)

print("For detailed logs, use a Code Server terminal, for example:")
print("docker logs -f nemollm-inference-microservice")
print("docker logs -f nemo-retriever-embedding-microservice")
print("docker logs -f nemo-retriever-ranking-microservice")


## Optional Health Checks

Run this after Compose shows the local NIM and Milvus containers as healthy.


In [ ]:
from urllib.request import urlopen

checks = [
    ("Nemotron 3 Nano", "http://127.0.0.1:8000/v1/health/ready"),
    ("Embedding NIM", "http://127.0.0.1:9080/v1/health/ready"),
    ("Reranking NIM", "http://127.0.0.1:1976/v1/health/ready"),
    ("GPU Milvus", "http://127.0.0.1:9091/healthz"),
    ("Unstructured retriever", "http://127.0.0.1:18086/health"),
]

for name, url in checks:
    try:
        with urlopen(url, timeout=10) as response:
            body = response.read(200).decode("utf-8", errors="replace")
        print(f"{name}: {response.status} {body}")
    except Exception as exc:
        print(f"{name}: not ready yet ({exc})")


## Prepare Sample Data

Download the product manuals so the ingestion notebook can load them into Milvus.


In [ ]:
from urllib.parse import unquote, urlparse
from urllib.request import urlretrieve

manual_list = REPO_ROOT / "data" / "list_manuals.txt"
manual_dir = REPO_ROOT / "data" / "manuals_pdf"
manual_dir.mkdir(parents=True, exist_ok=True)

downloaded = []
skipped = []

for url in manual_list.read_text(encoding="utf-8").splitlines():
    url = url.strip()
    if not url or url.startswith("#"):
        continue

    filename = Path(unquote(urlparse(url).path)).name
    target = manual_dir / filename
    if target.exists() and target.stat().st_size > 0:
        skipped.append(filename)
        continue

    print(f"Downloading {filename}...")
    urlretrieve(url, target)
    downloaded.append(filename)

print(f"Manuals ready in {manual_dir}")
print(f"Downloaded {len(downloaded)} file(s); skipped {len(skipped)} existing file(s).")


## Ingest Data

Open and run this notebook in Code Server:

```text
notebooks/ingest_data.ipynb
```

Choose the `AIVA LaunchPad` kernel if prompted. That kernel is configured to use the LaunchPad unstructured retriever host port `18086`.

The ingestion notebook loads:

- Product manuals and FAQ documents into GPU-backed Milvus collections
- Structured customer/order data into Postgres


## Open The User Interface

In the Code Server IDE, open the **Ports** tab next to the **Terminal** tab. Find port `3001` and open its forwarded URL.

The URL should look like:

```text
https://<launchpad-host>/coder/proxy/3001/
```

If port `3001` is not listed or is not healthy yet, wait another minute and refresh the **Ports** tab. The app waits on local NIM and database services during startup.
